# 🤖 Agenten: Emergentes Tool-Verhalten

Der Agent wechselt zwischen Nachdenken und Tool-Aufrufen. Aber die eigentliche Magie: **Die Optimierung verbessert nicht nur die Worte, sondern auch WIE der Agent seine Werkzeuge einsetzt** — die Entscheidungsstrategie!

## Was ist ein Agent?

Bisher haben unsere Modelle nur Text rein → Text raus gemacht. Ein **Agent** kann mehr: Er bekommt **Tools** (Taschenrechner, Datenbank-Suche, etc.) und entscheidet **selbst**, welches Tool er wann einsetzt.

Das Spannende: Dieses Verhalten ist **emergent** — wir programmieren nicht "wenn Mathe-Frage, dann Taschenrechner". Der Agent lernt das selbst.

In diesem Notebook:
1. Wir probieren die Tools einzeln aus
2. Wir lassen den Agenten selbst entscheiden
3. Wir optimieren sein Verhalten — ja, auch das geht!


In [ ]:
import sys; sys.path.insert(0, ".")
import dspy
from dspy_tasks.config import configure_dspy
from dspy_tasks.tasks import get_task, list_by_tier
from dspy_tasks.tools import TOOL_REGISTRY, calculate, search_tickets, get_ticket_stats
from dspy_tasks.actions import run_baseline, run_optimization
from dspy_tasks.visualize import *

MODEL = "github_copilot/gpt-5.1"
configure_dspy(MODEL)


## 🔧 Die verfügbaren Tools

Unsere Agenten haben Zugriff auf echte Tool-Funktionen. Lass uns sie erstmal direkt ausprobieren, bevor wir sie den Agenten geben.

### 🔄 Der Agent-Loop

So arbeitet ein Agent: Er **denkt nach** (was ist die Frage?), **wählt ein Tool** (brauche ich den Taschenrechner oder die Suche?), **führt es aus**, und **prüft das Ergebnis** (bin ich fertig oder muss ich nochmal nachdenken?).

Dieser Loop kann mehrere Runden dauern — bei komplexen Fragen kombiniert der Agent mehrere Tools nacheinander.


In [ ]:
diagram([
    {"label": "Denken", "detail": "Agent überlegt", "icon": "🤔", "color": "#0078d4"},
    {"label": "Tool wählen", "detail": "search? calculate?", "icon": "🔧", "color": "#ca5010"},
    {"label": "Ausführen", "detail": "Tool aufrufen", "icon": "⚡", "color": "#ca5010"},
    {"label": "Ergebnis prüfen", "detail": "Nochmal? Fertig?", "icon": "🔍", "color": "#107c10"},
], title="Der ReAct-Loop: Denken → Handeln → Beobachten")

### 🔧 Erst die Tools kennenlernen

Bevor wir sie dem Agenten geben, probieren wir die Tools selbst aus. So siehst du, was der Agent zur Verfügung hat:

- **calculate** — kann beliebige Mathe-Ausdrücke berechnen
- **search_tickets** — durchsucht die echten Ticket-Daten nach Stichworten
- **get_ticket_stats** — liefert Statistiken (z.B. Verteilung nach Priorität)

Klick Run und sieh dir die Ergebnisse an. Genau diese Ergebnisse bekommt nachher auch der Agent.


## 🧪 Tools direkt ausprobieren

Jedes Tool ist eine einfache Python-Funktion. Der Agent entscheidet selbst, welches er wann aufruft — aber lass uns sie erstmal einzeln testen.


In [ ]:
# Tools are just Python functions — try them!
calc_result = calculate("(45 * 3) + 17")
search_result = search_tickets("network")
stats_result = get_ticket_stats("priority")

print("🧮 calculate('(45 * 3) + 17'):")
print(f"   → {calc_result}")
print()
print("🔍 search_tickets('network'):")
print(f"   → {search_result}")
print()
print("📊 get_ticket_stats('priority'):")
print(f"   → {stats_result}")

## ⚙️ Modell konfigurieren

Wir verwenden ein voreingestelltes Modell. Du kannst es in der nächsten Zelle ändern.


## Task 16: Calculator Agent — Der einfachste Agent

Der Calculator Agent bekommt eine Mathe-Frage und hat ein `calculate`-Tool zur Verfügung. Er muss selber entscheiden, wann und wie er es benutzt.

In [ ]:
from dspy_tasks.config import get_available_models, get_default_model, configure_dspy
MODELS = get_available_models()


print(f"⏳ Running Calculator Agent on {MODEL}...")
result = run_baseline("calculator_agent", max_eval=5)
display_score("Calculator Agent", result.score)
display_results_table(result.individual_scores)


### 🔍 Warum ist der Search Agent schwieriger?

Der Calculator Agent hat ein einfaches Tool: Rechnung rein, Ergebnis raus. Der Search Agent muss:

1. Die richtige **Suchanfrage formulieren** (nicht trivial!)
2. Die **Ergebnisse interpretieren** (was ist relevant?)
3. Eventuell **nochmal suchen** mit anderen Stichworten
4. Alles zu einer **kohärenten Antwort** zusammenfassen

Das ist wie der Unterschied zwischen "Was ist 2+2?" und "Finde heraus, welches Team die meisten Netzwerk-Tickets hat."


## Task 17: Search & Synthesize — Deine Daten durchsuchen

Jetzt wird's interessant: der Agent durchsucht die echte Ticket-Datenbank und fasst Antworten zusammen. Er entscheidet selbst, welche Tools er braucht.

In [ ]:
print(f"⏳ Search Agent läuft...")
print("   Der Agent durchsucht die Ticket-Datenbank...\n")
result = run_baseline("search_agent", max_eval=5)
display_score("Search Agent", result.score)
display_results_table(result.individual_scores)


### ⚡ Agenten-Verhalten optimieren

Nicht nur die Worte werden besser — der Agent lernt auch, seine Tools **cleverer** einzusetzen. Der Optimizer verbessert:

- **Wann** der Agent welches Tool einsetzt (nicht immer sofort rechnen!)
- **Wie** er die Suchergebnisse interpretiert (relevante Teile filtern)
- **Ob** er nochmal nachfragt oder direkt antwortet (Qualität vs. Geschwindigkeit)

Wähl einen Agent-Task und einen Optimizer und schau dir den Unterschied an.


## 🎯 Agenten-Verhalten optimieren

Jetzt der Clou: wir optimieren nicht nur den Prompt, sondern auch **wie der Agent seine Werkzeuge einsetzt**. Die Optimierung lernt bessere Entscheidungsmuster!

Stell dir vor: Der Agent hat 5 Tool-Aufrufe gemacht und trotzdem die falsche Antwort geliefert. Nach der Optimierung: 2 gezielte Aufrufe und die richtige Antwort! Nicht nur der Prompt wird optimiert — auch WIE der Agent seine Tools einsetzt.

In [ ]:
from dspy_tasks.tasks import list_by_tier

# Verfügbare Agenten-Tasks:
print("Agenten-Aufgaben:")
for t in list_by_tier(4):
    print(f"  • {t.id}: {t.name}")

# Wähle einen Task (ändere den String):
AGENT_TASK = "calculator_agent"

task = get_task(AGENT_TASK)
print(f"\n⏳ Optimiere {task.name}...")
print("   Das kann eine Minute dauern...\n")
result = run_optimization(AGENT_TASK, "BootstrapFewShot", max_eval=5)
display_improvement(result.baseline_score, result.optimized_score)
display_prompt_diff(result.prompt_before, result.prompt_after)

display_insight("Agenten-Optimierung",
    f"Der Optimizer hat den Agenten von {result.baseline_score:.0%} auf "
    f"{result.optimized_score:.0%} verbessert. Nicht nur Worte — "
    "auch die Entscheidungsstrategie für Tool-Nutzung wurde optimiert.")


### 📝 Was du mitnehmen solltest

1. **Agenten = LLM + Tools** — Das Modell entscheidet selbst, welches Tool es nutzt
2. **Emergentes Verhalten** — Wir programmieren keine Regeln, der Agent lernt sie
3. **Auch Agent-Verhalten ist optimierbar** — Nicht nur was der Agent sagt, sondern wie er vorgeht

Im letzten Notebook bringen wir alles zusammen!


## ⏭️ Das Finale!

Agenten können Tools nutzen UND ihr Verhalten kann optimiert werden. Du hast jetzt alle Bausteine gesehen:

1. ✅ Metriken und Evaluation
2. ✅ Automatische Prompt-Optimierung
3. ✅ Domain-Daten als Wettbewerbsvorteil
4. ✅ Agenten-Optimierung

**Zeit für das Gesamtbild und das Quiz!** → Notebook 05

👉 **[Weiter zu Notebook: Das Gesamtbild →](05_full_picture.ipynb)**
